In [ ]:
import matplotlib
matplotlib.use('TkAgg')  # Configurar el backend de Matplotlib
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.path import Path
from matplotlib.patches import PathPatch
import tkinter as tk
from tkinter import ttk, messagebox
from xml.dom import minidom
import time
import os


class SVGLoader:
    """Clase para cargar contornos desde archivos SVG."""

    @staticmethod
    def _circle_to_polygon(cx, cy, r, num_points=100):
        """Convierte un círculo en un polígono con un número específico de puntos."""
        points = []
        for i in range(num_points):
            angle = 2 * np.pi * i / num_points
            x = cx + r * np.cos(angle)
            y = cy + r * np.sin(angle)
            points.append((x, y))
        return points

    @staticmethod
    def _parse_path(d):
        """Convierte un elemento <path> en una lista de puntos."""
        # Esta es una implementación básica. Puedes mejorarla para manejar todos los casos.
        points = []
        commands = d.split()
        i = 0
        while i < len(commands):
            cmd = commands[i]
            if cmd == 'M':  # Moveto
                x = float(commands[i + 1])
                y = float(commands[i + 2])
                points.append((x, y))
                i += 3
            elif cmd == 'L':  # Lineto
                x = float(commands[i + 1])
                y = float(commands[i + 2])
                points.append((x, y))
                i += 3
            elif cmd == 'C':  # Curveto (ignoramos las curvas por simplicidad)
                i += 7
            else:
                i += 1
        return points

    @staticmethod
    def load_svg_contour(file_path):
        """
        Carga contornos y huecos desde un archivo SVG.

        Args:
            file_path (str): Ruta al archivo SVG.

        Returns:
            list: Lista de contornos, donde el primer elemento es el contorno exterior
                  y los siguientes son los huecos.
        """
        try:
            doc = minidom.parse(file_path)
            contours = []

            # Cargar polígonos
            polygons = doc.getElementsByTagName('polygon')
            for polygon in polygons:
                points = polygon.getAttribute('points').strip().split()
                contour = [tuple(map(float, point.split(','))) for point in points]
                contours.append(contour)

            # Cargar círculos y convertirlos en polígonos
            circles = doc.getElementsByTagName('circle')
            for circle in circles:
                cx = float(circle.getAttribute('cx'))
                cy = float(circle.getAttribute('cy'))
                r = float(circle.getAttribute('r'))
                contour = SVGLoader._circle_to_polygon(cx, cy, r)
                contours.append(contour)

            # Cargar paths (rutas)
            paths = doc.getElementsByTagName('path')
            for path in paths:
                d = path.getAttribute('d')
                contour = SVGLoader._parse_path(d)
                if contour:
                    contours.append(contour)

            doc.unlink()
            return contours
        except Exception as e:
            raise ValueError(f"Error al cargar el archivo SVG: {e}")


class Simulator:
    """Clase para simular la impresión 3D de un modelo SVG."""

    def __init__(self, contours, resolution, speed):
        """
        Inicializa la simulación.

        Args:
            contours (list): Lista de contornos (exterior y huecos).
            resolution (float): Resolución de la simulación en mm.
            speed (int): Velocidad de la simulación.
        """
        self.contours = contours
        self.resolution = resolution
        self.speed = speed
        self.exterior = Path(contours[0])  # Contorno exterior
        self.holes = [Path(hole) for hole in contours[1:]]  # Lista de huecos

        # Generar puntos dentro del contorno exterior
        self.x = np.arange(min(p[0] for p in contours[0]), max(p[0] for p in contours[0]), resolution)
        self.y = np.arange(min(p[1] for p in contours[0]), max(p[1] for p in contours[0]), resolution)
        self.X, self.Y = np.meshgrid(self.x, self.y)
        self.all_points = np.c_[self.X.flatten(), self.Y.flatten()]

        # Filtrar puntos: dentro del exterior y fuera de los huecos
        self.inside_points = self.all_points[
            self.exterior.contains_points(self.all_points, radius=-0.01) &
            ~np.any([hole.contains_points(self.all_points, radius=-0.01) for hole in self.holes], axis=0)
        ]
        self.sorted_points = self.inside_points[np.argsort(self.inside_points[:, 1])]
        self.columns = np.unique(self.sorted_points[:, 0])
        self.trajectory = self._generate_trajectory()
        self.total_points = len(self.trajectory)
        self.current_speed = speed

    def _generate_trajectory(self):
        """Genera la trayectoria de impresión."""
        trajectory = []
        for i, column in enumerate(self.columns):
            column_points = self.sorted_points[self.sorted_points[:, 0] == column]
            if len(column_points) > 0:
                trajectory.append(column_points if i % 2 == 0 else column_points[::-1])
        return np.vstack(trajectory)

    def simulate(self):
        """Ejecuta la simulación."""
        fig, ax = plt.subplots()
        fig.patch.set_facecolor('white')
        ax.set_facecolor('white')

        # Dibujar contorno exterior y huecos
        ax.plot(*zip(*self.contours[0]), 'b-', linewidth=0.5, label="Contorno")
        for hole in self.contours[1:]:
            ax.plot(*zip(*hole), 'b-', linewidth=0.5)
        ax.set_aspect('equal')
        ax.set_xlim(min(p[0] for p in self.contours[0]), max(p[0] for p in self.contours[0]))
        ax.set_ylim(min(p[1] for p in self.contours[0]), max(p[1] for p in self.contours[0]))
        ax.legend()

        # Máscara de recorte para el contorno exterior
        patch = PathPatch(self.exterior, facecolor='none', edgecolor='none')
        ax.add_patch(patch)

        # Elementos gráficos
        printed_points, = ax.plot([], [], 'o', markersize=2.5, color='orange')
        line, = ax.plot([], [], '-', linewidth=0.5, color='#404040')
        current_dot, = ax.plot([], [], 'go', markersize=3)

        # Aplicar máscara a todos los elementos
        printed_points.set_clip_path(patch)
        line.set_clip_path(patch)
        current_dot.set_clip_path(patch)

        # Información de la simulación
        info_text = ax.text(0.02, 0.98, "", transform=ax.transAxes, verticalalignment='top', fontsize=10, color='black')
        ax_speed = plt.axes([0.2, 0.05, 0.6, 0.03], facecolor='lightgoldenrodyellow')
        speed_slider = plt.Slider(ax_speed, 'Velocidad', 1, 100, valinit=self.speed, valstep=1)

        def update_speed(val):
            self.current_speed = val
        speed_slider.on_changed(update_speed)

        plt.show(block=False)
        start_time = time.perf_counter()
        last_frame_time = start_time

        try:
            for frame in range(len(self.trajectory)):
                current_trajectory = self.trajectory[:frame + 1]
                printed_points.set_data(current_trajectory.T)
                line.set_data(current_trajectory.T)
                current_dot.set_data(*current_trajectory[frame])

                elapsed_time = time.perf_counter() - start_time
                info_text.set_text(f"Puntos: {frame + 1}/{self.total_points}\nTiempo: {elapsed_time:.2f}s")

                fig.canvas.draw_idle()
                fig.canvas.flush_events()

                target_delay = 1 / self.current_speed
                elapsed = time.perf_counter() - last_frame_time
                if (remaining := target_delay - elapsed) > 0:
                    time.sleep(remaining * 0.95)

                last_frame_time = time.perf_counter()
                plt.pause(0.01)

        except Exception as e:
            print(f"Error durante la simulación: {e}")


class App:
    """Clase principal de la interfaz gráfica."""

    def __init__(self, root):
        self.root = root
        self.root.title("Simulador de Impresión 3D")
        self.root.geometry("800x600")
        self.root.configure(bg='#2C3E50')
        self._setup_ui()

    def __init__(self, root):
        self.root = root
        self.root.title("Simulador de Impresión 3D")
        self.root.geometry("800x600")
        self.root.configure(bg='#1E1E1E')  # Fondo oscuro
        self._setup_ui()

    def _setup_ui(self):
        """Configura la interfaz gráfica."""
        main_frame = tk.Frame(self.root, bg='#1E1E1E')
        main_frame.pack(expand=True, fill='both', padx=40, pady=40)

        # Título principal
        title_label = tk.Label(main_frame, text="🖨️ SIMULADOR DE IMPRESIÓN 3D",
                               font=('Arial', 26, 'bold'), bg='#1E1E1E', fg='#00A8E8')
        title_label.pack(pady=20)
        
        # Frame para la imagen
        image_frame = tk.Frame(main_frame, bg='#1E1E1E')
        image_frame.pack(pady=20)

        # Cargar y mostrar la imagen
        try:
            # Asumiendo que la imagen está en la carpeta 'assets'
            image_path = os.path.join("assets", "image.png")
            # Cargar la imagen usando PIL
            from PIL import Image, ImageTk
            image = Image.open(image_path)
            # Redimensionar la imagen manteniendo la proporción
            width = 500  # Ancho deseado
            ratio = width / image.size[0]
            height = int(image.size[1] * ratio)
            image = image.resize((width, height), Image.Resampling.LANCZOS)
            photo = ImageTk.PhotoImage(image)
            
            # Crear y mostrar el label con la imagen
            image_label = tk.Label(image_frame, image=photo, bg='#1E1E1E')
            image_label.image = photo  # Mantener una referencia
            image_label.pack()
        except Exception as e:
            print(f"Error al cargar la imagen: {e}")

        # Nombres del grupo
        group_label = tk.Label(main_frame, text="Grupo: Wellington Barros, Hodalys Lopez, Josue Mantuano, Jonathan Paredes",
                               font=('Arial', 12, 'italic'), bg='#1E1E1E', fg='#BBBBBB')
        group_label.pack(pady=5)
        

        # Descripción del proyecto
        desc_label = tk.Label(main_frame, text=(
            "Este proyecto simula el proceso de impresión 3D utilizando modelos SVG.\n"
            "Permite cargar archivos SVG y visualizar cómo serían impresos capa por capa.\n"
            "El objetivo es entender el funcionamiento básico de una impresora 3D."
        ), font=('Arial', 12), bg='#1E1E1E', fg='#CCCCCC', justify="center", wraplength=700)
        desc_label.pack(pady=10)

        # Descripción
        desc_label = tk.Label(main_frame, text="Simula la impresión 3D de modelos SVG con precisión.",
                              font=('Arial', 12), bg='#1E1E1E', fg='#CCCCCC')
        desc_label.pack(pady=10)

        # Botón de inicio
        style = ttk.Style()
        style.configure('Accent.TButton', font=('Arial', 14), background='#00A8E8', foreground='white',
                        padding=10, borderwidth=0)
        style.map('Accent.TButton', background=[('active', '#0077B6')], relief=[('pressed', 'sunken')])

        start_button = ttk.Button(main_frame, text="🚀 Iniciar Simulación", style='Accent.TButton',
                                  command=self._show_input_window)
        start_button.pack(pady=30)
        
    def _show_input_window(self):
        """Muestra la ventana de parámetros de impresión."""
        input_win = tk.Toplevel(self.root)
        input_win.title("Parámetros de Impresión")
        input_win.configure(bg='#34495E')

        main_frame = tk.Frame(input_win, bg='#34495E')
        main_frame.pack(pady=20, padx=20, fill='both', expand=True)

        # Obtener lista de SVGs disponibles
        svg_files = []
        if os.path.exists("models"):
            svg_files = [f[:-4] for f in os.listdir("models") if f.endswith('.svg')]

        # Título de la segunda ventana
        title_label = tk.Label(main_frame, text="Selecciona un modelo SVG", font=('Arial', 16, 'bold'),
                               bg='#34495E', fg='#ECF0F1')
        title_label.pack(pady=10)

        # Frame para los checkbuttons
        check_frame = tk.Frame(main_frame, bg='#34495E')
        check_frame.pack(pady=10)

        # Variables para los checkbuttons
        self.selected_model = tk.StringVar()

        # Crear checkbuttons para cada modelo SVG
        for svg_file in svg_files:
            check = tk.Radiobutton(check_frame, text=svg_file, font=('Arial', 12), bg='#34495E', fg='#ECF0F1',
                                   selectcolor='#34495E', activebackground='#34495E', activeforeground='#3498DB',
                                   variable=self.selected_model, value=svg_file)
            check.pack(anchor='w', pady=5)

        # Etiquetas y campos de entrada
        tk.Label(main_frame, text="Resolución (mm):", font=('Arial', 12), bg='#34495E', fg='#ECF0F1').pack(pady=5)
        self.res_entry = tk.Entry(main_frame, font=('Arial', 12), width=50)
        self.res_entry.insert(0, " ")
        self.res_entry.pack(pady=50)

        # Botón de inicio
        start_button = tk.Button(main_frame, text="Iniciar Simulación", font=('Arial', 12), bg='#3498DB', fg='white',
                                 activebackground='#2980B9', activeforeground='white',
                                 command=lambda: self.start_process(input_win))
        start_button.pack(pady=2)

    def start_process(self, window):
        """Inicia la simulación con los parámetros seleccionados."""
        model_name = self.selected_model.get()
        resolution = self.res_entry.get()

        if not model_name:
            messagebox.showerror("Error", "Debes seleccionar un modelo SVG.")
            return
        try:
            resolution = float(resolution)
            if resolution <= 0:
                raise ValueError("La resolución debe ser un número positivo.")
        except ValueError as e:
            messagebox.showerror("Error", f"Resolución inválida: {e}")
            return

        try:
            file_path = os.path.join("models", f"{model_name}.svg")
            contours = SVGLoader.load_svg_contour(file_path)
            simulator = Simulator(contours, resolution, speed=1)
            window.destroy()
            simulator.simulate()
        except Exception as e:
            messagebox.showerror("Error", str(e))


if __name__ == "__main__":
    root = tk.Tk()
    style = ttk.Style(root)
    style.theme_use('clam')
    style.configure('Accent.TButton', font=('Arial', 14), background='#3498DB', foreground='white', padding=100)
    style.map('Accent.TButton', background=[('active', '#2980B9')])
    App(root)
    root.mainloop()